In [ ]:
import os
import re
import json
import time
import requests
from tqdm.auto import tqdm
from bs4 import BeautifulSoup
from dotenv import load_dotenv

load_dotenv()

DANISH_LEGISLATION_LIST_FILE = os.getenv("DANISH_LEGISLATION_LIST_FILE", "danish_tax_legislation.txt")
DANISH_JSON_OUTPUT_DIR = os.getenv("DANISH_JSON_OUTPUT_DIR", "danish_json_out")
REQUEST_DELAY = float(os.getenv("REQUEST_DELAY", "2.0"))  # seconds between requests
ELI_BASE = "https://www.retsinformation.dk/eli/lta"

In [ ]:
class DanishLegislationCrawler:

    # Citation patterns for Danish legislation text
    _CITATION_PATTERNS = [
        # Named act + paragraph: "personskattelovens § 2" or "ligningsloven §§ 7 og 8"
        re.compile(r'(\b\w+loven[s]?\s+(?:§§?\s*[\d\w,\s]+|kapitel\s+\d+))', re.IGNORECASE),
        # Act by number and date: "lov nr. 799 af 7. august 2019"
        re.compile(r'(lov(?:bekendtgørelse)?\s+nr\.\s+\d+\s+af\s+[\d.]+\s+\w+\s+\d{4})', re.IGNORECASE),
        # Bekendtgørelse by number
        re.compile(r'(bekendtgørelse\s+nr\.\s+\d+\s+af\s+[\d.]+\s+\w+\s+\d{4})', re.IGNORECASE),
    ]

    _MAX_XML_BYTES = 5 * 1024 * 1024  # 5 MB — skip larger files to avoid OOM crash

    def __init__(self):
        os.makedirs(".cache_dk", exist_ok=True)
        os.makedirs(DANISH_JSON_OUTPUT_DIR, exist_ok=True)

    # ------------------------------------------------------------------
    # Fetch
    # ------------------------------------------------------------------

    def _eli_uri(self, year, number):
        return f"{ELI_BASE}/{year}/{number}"

    def _fetch_xml(self, year, number):
        cache = os.path.join(".cache_dk", f"{year}_{number}.xml")
        if os.path.exists(cache):
            if os.path.getsize(cache) > self._MAX_XML_BYTES:
                print(f"  Skipping {year}/{number}: cached file exceeds {self._MAX_XML_BYTES // 1024 // 1024} MB")
                return None, True
            with open(cache, "rb") as f:
                return BeautifulSoup(f.read(), "xml"), True
        url = f"{ELI_BASE}/{year}/{number}/xml"
        try:
            resp = requests.get(url, timeout=30, stream=True)
            resp.raise_for_status()
            content = b""
            for chunk in resp.iter_content(chunk_size=65536):
                content += chunk
                if len(content) > self._MAX_XML_BYTES:
                    print(f"  Skipping {year}/{number}: response exceeds {self._MAX_XML_BYTES // 1024 // 1024} MB")
                    return None, False
            with open(cache, "wb") as f:
                f.write(content)
            return BeautifulSoup(content, "xml"), False
        except requests.RequestException as e:
            print(f"  Failed to fetch {url}: {e}")
            return None, False

    # ------------------------------------------------------------------
    # Text extraction
    # ------------------------------------------------------------------

    def _extract_stk_text(self, stk):
        """Extract text content from a <Stk>, skipping its <Explicatus> label."""
        parts = []
        for child in stk.children:
            if not hasattr(child, "name") or child.name is None:
                continue
            if child.name == "Explicatus":
                continue
            text = child.get_text(separator=" ", strip=True)
            if text:
                parts.append(text)
        return " ".join(parts).strip()

    # ------------------------------------------------------------------
    # Citation extraction
    # ------------------------------------------------------------------

    def _extract_citations(self, text):
        """Extract citation references from plain text. Returns list of citation dicts."""
        citations = []
        seen = set()
        for pattern in self._CITATION_PATTERNS:
            for match in pattern.finditer(text):
                citation_text = match.group(1).strip()
                if citation_text not in seen:
                    seen.add(citation_text)
                    citations.append({
                        "id": f"cit_{abs(hash(citation_text)) % 10**8}",
                        "uri": None,
                        "title": None,
                        "class": None,
                        "year": None,
                        "number": None,
                        "text": citation_text,
                    })
        return citations

    # ------------------------------------------------------------------
    # Hierarchy parsing: Stk → Paragraf → Kapitel → Afsnit
    # ------------------------------------------------------------------

    def _stks_to_paragraphs(self, paragraf, section_uri):
        """Convert <Stk> children of a <Paragraf> into paragraph dicts."""
        stks = paragraf.find_all("Stk", recursive=False)
        if not stks:
            return []
        paragraphs = []
        for order, stk in enumerate(stks, start=1):
            explicatus = stk.find("Explicatus", recursive=False)
            stk_num = None
            if explicatus:
                raw = explicatus.get_text(strip=True)
                m = re.search(r'Stk\.\s*(\d+)', raw)
                stk_num = m.group(1) if m else raw.rstrip(".")
            else:
                stk_num = "1"  # single unnamed stk

            text = self._extract_stk_text(stk)
            citations = self._extract_citations(text)

            paragraphs.append({
                "order": order,
                "paragraph_number": stk_num,
                "text": text,
                "uri": f"{section_uri}#stk_{stk_num or order}",
                "restrict_start_date": None,
                "restrict_end_date": None,
                "restrict_extent": None,
                "status": None,
                "commentaries": [{
                    "ref_id": f"cite_{c['id']}",
                    "type": "citation",
                    "text": c["text"],
                    "citations": [c],
                    "citation_subrefs": [],
                } for c in citations] if citations else [],
            })
        return paragraphs

    def _paragraf_to_section(self, paragraf, order, parent_uri, rubrica_title=None):
        """Convert a <Paragraf> element into a section dict."""
        local_id = paragraf.get("id") or paragraf.get("localId") or str(order)
        section_uri = f"{parent_uri}#paragraf_{local_id}"

        explicatus = paragraf.find("Explicatus", recursive=False)
        raw_label = explicatus.get_text(strip=True) if explicatus else f"§ {local_id}"
        # Capture "§ 9 C." correctly — number and optional letter suffix (with possible space)
        m = re.search(r'§\s*(\d+)\s*([A-Za-z]?)', raw_label)
        if m:
            section_num = m.group(1) + (" " + m.group(2) if m.group(2) else "")
        else:
            section_num = raw_label

        paragraphs = self._stks_to_paragraphs(paragraf, section_uri)

        section = {
            "order": order,
            "section_number": section_num,
            "title": rubrica_title,
            "uri": section_uri,
            "restrict_start_date": None,
            "restrict_end_date": None,
            "restrict_extent": None,
            "status": None,
            "commentaries": [],
            "paragraphs": paragraphs,
        }
        if not paragraphs:
            # No Stk children — store full text at section level
            section["text"] = paragraf.get_text(separator=" ", strip=True)
        return section

    def _collect_sections_from(self, container, parent_uri):
        """Collect sections from a container that may have ParagrafGruppe and/or direct Paragraf."""
        sections = []
        order = 0
        for child in container.children:
            if not hasattr(child, "name") or child.name is None:
                continue
            if child.name == "ParagrafGruppe":
                rubrica = child.find("Rubrica", recursive=False)
                rubrica_text = rubrica.get_text(strip=True) if rubrica else None
                for paragraf in child.find_all("Paragraf", recursive=False):
                    order += 1
                    sections.append(self._paragraf_to_section(paragraf, order, parent_uri, rubrica_text))
            elif child.name == "Paragraf":
                order += 1
                sections.append(self._paragraf_to_section(child, order, parent_uri))
        return sections

    def _kapitel_to_chapter(self, kapitel, order, parent_uri):
        """Convert a <Kapitel> element into a chapter dict."""
        local_id = kapitel.get("id") or kapitel.get("localId") or str(order)
        chapter_uri = f"{parent_uri}#kapitel_{local_id}"

        title_tag = kapitel.find(["Overskrift", "Rubrica", "Titel"], recursive=False)
        title = title_tag.get_text(strip=True) if title_tag else None

        sections = self._collect_sections_from(kapitel, chapter_uri)

        return {
            "order": order,
            "chapter_number": local_id,
            "uri": chapter_uri,
            "restrict_start_date": None,
            "restrict_end_date": None,
            "status": None,
            "title": title,
            "sections": sections,
        }

    def _find_bog(self, soup):
        """Return the <Bog> element that contains legislation content (may be second if first is empty)."""
        for bog in soup.find_all("Bog"):
            if any(hasattr(c, "name") and c.name for c in bog.children):
                return bog
        return soup.find("DokumentIndhold") or soup

    def _extract_parts(self, soup, doc_uri):
        """Extract the full Part → Chapter → Section → Paragraph hierarchy."""
        bog = self._find_bog(soup)

        # --- Case 1: Act has <Afsnit> (Parts) ---
        afsnitts = bog.find_all("Afsnit", recursive=False)
        if afsnitts:
            parts = []
            for part_order, afsnit in enumerate(afsnitts, start=1):
                local_id = afsnit.get("id") or afsnit.get("localId") or str(part_order)
                part_uri = f"{doc_uri}#afsnit_{local_id}"

                title_tag = afsnit.find(["Overskrift", "Titel"], recursive=False)
                title = title_tag.get_text(strip=True) if title_tag else None

                chapters = []
                for chap_order, kapitel in enumerate(afsnit.find_all("Kapitel", recursive=False), start=1):
                    chapters.append(self._kapitel_to_chapter(kapitel, chap_order, part_uri))

                # Afsnit with no Kapitel — treat paragraphs as a synthetic chapter
                if not chapters:
                    synth_uri = f"{part_uri}#kapitel_1"
                    sections = self._collect_sections_from(afsnit, synth_uri)
                    if sections:
                        chapters.append({
                            "order": 1, "chapter_number": None, "uri": synth_uri,
                            "restrict_start_date": None, "restrict_end_date": None,
                            "status": None, "title": None, "sections": sections,
                        })

                parts.append({
                    "order": part_order,
                    "part_number": local_id,
                    "uri": part_uri,
                    "restrict_start_date": None,
                    "restrict_end_date": None,
                    "status": None,
                    "title": title,
                    "chapters": chapters,
                })
            return parts

        # --- Case 2: No Afsnit — Kapitel directly under Bog ---
        synth_part_uri = f"{doc_uri}#part_1"
        kapitels = bog.find_all("Kapitel", recursive=False)
        if kapitels:
            chapters = [self._kapitel_to_chapter(k, i, synth_part_uri) for i, k in enumerate(kapitels, 1)]
            return [{
                "order": 1, "part_number": None, "uri": synth_part_uri,
                "restrict_start_date": None, "restrict_end_date": None,
                "status": None, "title": None, "chapters": chapters,
            }]

        # --- Case 3: Flat list of ParagrafGruppe / Paragraf ---
        synth_chap_uri = f"{synth_part_uri}#kapitel_1"
        sections = self._collect_sections_from(bog, synth_chap_uri)
        return [{
            "order": 1, "part_number": None, "uri": synth_part_uri,
            "restrict_start_date": None, "restrict_end_date": None,
            "status": None, "title": None,
            "chapters": [{
                "order": 1, "chapter_number": None, "uri": synth_chap_uri,
                "restrict_start_date": None, "restrict_end_date": None,
                "status": None, "title": None, "sections": sections,
            }],
        }]

    # ------------------------------------------------------------------
    # Top-level parse
    # ------------------------------------------------------------------

    def parse(self, year, number):
        """Fetch and parse one act. Returns the standard JSON dict or None on failure."""
        try:
            soup, from_cache = self._fetch_xml(year, number)
            if not soup:
                return None, from_cache

            doc_uri = self._eli_uri(year, number)
            meta = soup.find("Meta")

            def _text(tag_name):
                tag = meta.find(tag_name) if meta else None
                return tag.get_text(strip=True) if tag else None

            raw_type = _text("DocumentType") or ""
            doc_type = raw_type.split()[0] if raw_type else None

            parts = self._extract_parts(soup, doc_uri)

            return {
                "legislation_url": doc_uri,
                "identifier": {
                    "title": _text("DocumentTitle"),
                    "description": _text("PopularTitle"),
                    "publisher": _text("Ministry"),
                    "modified": _text("DiesSigni") or _text("DiesEdicti"),
                    "uri": doc_uri,
                    "valid_date": _text("EndDate"),
                },
                "super": {
                    "supersedes": None,
                    "superseded_by": None,
                },
                "metadata": {
                    "year": _text("Year") or str(year),
                    "number": _text("Number") or str(number),
                    "enactment_date": _text("DiesSigni"),
                    "status": _text("Status"),
                    "isbn": None,
                    "category": doc_type,
                    "coming_into_force": _text("StartDate"),
                    "unapplied_effects": [],
                },
                "parts": parts,
                "schedules": [],
                "explanatory_notes": None,
            }, from_cache
        except Exception as e:
            print(f"  Error parsing {year}/{number}: {e}")
            return None, False

    # ------------------------------------------------------------------
    # Crawl loop
    # ------------------------------------------------------------------

    def crawl(self):
        with open(DANISH_LEGISLATION_LIST_FILE) as f:
            seeds = [
                line.strip() for line in f
                if line.strip() and not line.strip().startswith("#")
            ]

        print(f"Processing {len(seeds)} acts from {DANISH_LEGISLATION_LIST_FILE}")

        for seed in tqdm(seeds, desc="Crawling Danish legislation"):
            parts = seed.split("/")
            if len(parts) != 2:
                print(f"  Skipping invalid entry: {seed}")
                continue

            year, number = parts
            doc, from_cache = self.parse(year, number)
            if not doc:
                continue

            year_dir = os.path.join(DANISH_JSON_OUTPUT_DIR, str(year))
            os.makedirs(year_dir, exist_ok=True)
            output_path = os.path.join(year_dir, f"{year}_{number}.json")

            with open(output_path, "w", encoding="utf-8") as f:
                json.dump(doc, f, indent=4, ensure_ascii=False)

            title = doc["identifier"].get("title") or "N/A"
            n_parts = len(doc["parts"])
            n_sections = sum(
                len(ch["sections"])
                for p in doc["parts"] for ch in p["chapters"]
            )
            n_paras = sum(
                len(s["paragraphs"])
                for p in doc["parts"] for ch in p["chapters"] for s in ch["sections"]
            )
            print(f"  {year}/{number}  {title[:70]}")
            print(f"    → {n_parts} part(s), {n_sections} section(s), {n_paras} paragraph(s) — saved to {output_path}")

            if not from_cache:
                time.sleep(REQUEST_DELAY)

In [ ]:
# Quick smoke test: parse one known-good act and inspect the structure
crawler = DanishLegislationCrawler()
doc = crawler.parse("2019", "799")  # Personskatteloven

if doc:
    print("Title  :", doc["identifier"]["title"])
    print("Popular:", doc["identifier"]["description"])
    print("Status :", doc["metadata"]["status"])
    print("Enacted:", doc["metadata"]["enactment_date"])
    print("Parts  :", len(doc["parts"]))
    for p in doc["parts"][:3]:
        print(f"  Part {p['part_number']}: {p['title']} — {len(p['chapters'])} chapter(s)")
        for ch in p["chapters"][:2]:
            n_sec = len(ch["sections"])
            n_para = sum(len(s["paragraphs"]) for s in ch["sections"])
            print(f"    Chapter {ch['chapter_number']}: {ch['title']} — {n_sec} section(s), {n_para} paragraph(s)")
            for s in ch["sections"][:2]:
                print(f"      § {s['section_number']}: {s['title']} — {len(s['paragraphs'])} stk")
                for para in s["paragraphs"][:1]:
                    print(f"        Stk {para['paragraph_number']}: {para['text'][:100]}...")
else:
    print("Parse failed — check network access and that lxml is installed.")

In [ ]:
# Run the full crawl over all acts in the seed list
crawler = DanishLegislationCrawler()
crawler.crawl()

In [ ]:
# Inspect the saved JSON for any act
import glob

files = sorted(glob.glob(f"{DANISH_JSON_OUTPUT_DIR}/**/*.json", recursive=True))
print(f"Output files ({len(files)} total):")
for f in files:
    with open(f) as fh:
        d = json.load(fh)
    n_sec = sum(len(ch["sections"]) for p in d["parts"] for ch in p["chapters"])
    n_para = sum(len(s["paragraphs"]) for p in d["parts"] for ch in p["chapters"] for s in ch["sections"])
    print(f"  {f}  →  {n_sec} sections, {n_para} paragraphs")